# Gamelog pipeline — Bronze → Silver

**Bronze** (`raw.*` in Supabase): fetch from NBA API → 5 tables  
**Silver** (`silver.player_gamelogs`): merge + positions + name canon + rotowire → upload

Run cells top-to-bottom. Set `SEASON` and `SEASON_TYPE` once in the config cell.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

pd.set_option("display.max_columns", None)

In [ ]:
# ── change these ──────────────────────────────────────────────────────────
SEASON = "2019-20"
SEASON_TYPE = "Regular Season"          #"Regular Season" | "Playoffs"
SKIP_ROTOWIRE = False             # True if you don't have the rotowire CSV yet
RUN_TRACKING_FETCH = False        # True to (re)fetch start_positions (slow)
UPLOAD_TO_SUPABASE = True         # False to build silver in memory only
# ─────────────────────────────────────────────────────────────────────────

## 1. Bronze — fetch NBA API → `raw.*`

| Function | Table | Source |
|---|---|---|
| `fetch_all(season, season_type)` | all 4 below | one call |
| `fetch_player_base` | `raw.player_base` | PlayerGameLogs Base |
| `fetch_player_adv` | `raw.player_adv` | PlayerGameLogs Advanced |
| `fetch_team_base` | `raw.team_base` | TeamGameLogs Base |
| `fetch_team_adv` | `raw.team_adv` | TeamGameLogs Advanced |
| `fetch_boxscoreplayertrackv3` | `raw.start_positions` | per-game tracking (slow) |

Requires `SUPABASE_DB_URL` in `.env`. Safe to re-run — upserts on natural keys.

In [ ]:
from src.utils.bronze import fetch_all, fetch_boxscoreplayertrackv3

print(f"Bronze: {SEASON} {SEASON_TYPE}")
fetch_all(SEASON, SEASON_TYPE)

if RUN_TRACKING_FETCH:
    fetch_boxscoreplayertrackv3(
        SEASON,
        SEASON_TYPE,
        batch_size=100,
        delay=2.5,
        workers=5,
    )

Bronze: 2019-20 Regular Season
    … 10,000/22,393 rows
    … 20,000/22,393 rows
    … 22,393/22,393 rows
  ✓ raw.player_base — 22,393 rows upserted (postgres)
    … 10,000/22,393 rows
    … 20,000/22,393 rows
    … 22,393/22,393 rows
  ✓ raw.player_adv — 22,393 rows upserted (postgres)
    … 2,118/2,118 rows
  ✓ raw.team_base — 2,118 rows upserted (postgres)
    … 2,118/2,118 rows
  ✓ raw.team_adv — 2,118 rows upserted (postgres)


: 

In [74]:
from src.utils.db import read_df

game_filter = "002%" if SEASON_TYPE == "Regular Season" else "004%"
params = {"season": SEASON, "prefix": game_filter}
where = "season_year = %(season)s AND game_id LIKE %(prefix)s"
sp_where = (
    "game_id IN (SELECT DISTINCT game_id FROM raw.player_base "
    f"WHERE season_year = %(season)s AND game_id LIKE %(prefix)s)"
)

print(f"{'table':<18} rows")
print("-" * 26)
for table in ("player_base", "player_adv", "team_base", "team_adv", "start_positions"):
    w = sp_where if table == "start_positions" else where
    n = len(read_df(table, where=w, params=params))
    flag = "  ⚠ empty" if table == "team_adv" and n == 0 else ""
    print(f"{table:<18} {n:>6,}{flag}")

table              rows
--------------------------
player_base        23,054
player_adv         23,054
team_base           2,160
team_adv            2,160
start_positions    28,859


## 2. Silver — merge `raw.*` → `silver.player_gamelogs`

| Function | What it does |
|---|---|
| `build_gamelogs_silver(season, season_type)` | read bronze + positions + name canon + rotowire |
| `upsert_silver(df, season_type=...)` | upload to `silver.player_gamelogs` |

- `IS_PLAYOFF` is set automatically from `season_type`
- Re-upsert updates existing rows on `(game_id, player_id)` — no need to drop the table

In [75]:
from importlib import reload
import src.utils.silver as silver
import src.utils.db as db
reload(silver)
reload(db)

from src.utils.silver import build_gamelogs_silver
from src.utils.db import upsert_silver

df = build_gamelogs_silver(
    SEASON,
    SEASON_TYPE,
    skip_rotowire=SKIP_ROTOWIRE,
)

df[["PLAYER_NAME", "GAME_DATE", "MATCHUP", "PTS", "TEAM_SPREAD", "GAME_TOTAL", "IS_PLAYOFF"]].head()

── Silver: 2020-21 Regular Season ──
  loading raw.* from Supabase…
  read raw.player_base — 23,054 rows
  read raw.player_adv — 23,054 rows
  read raw.team_base — 2,160 rows
  read raw.team_adv — 2,160 rows
  read raw.start_positions — 28,859 rows
  rotowire — API season '2020' from rotowire_nba_2020.csv
  merged — (23054, 183)
  names — 540 unique | missing vs reference: 326
  missing sample: ['Abdel Nader', 'Adam Mokoka', 'Al-Farouq Aminu', 'Alec Burks', 'Aleksej Pokusevski', 'Alen Smailagic', 'Alex Len', 'Alfonzo McKinnie', 'Alize Johnson', 'Amida Brimah', 'Anderson Varejao', 'Andre Iguodala', 'Andre Roberson', 'Anthony Lamb', 'Anthony Tolliver', 'Anžejs Pasečņiks', 'Armoni Brooks', 'Aron Baynes', 'Ashton Hagans', 'Austin Rivers']
✓ Silver frame — 23,054 rows, 188 columns


,PLAYER_NAME,GAME_DATE,MATCHUP,PTS,TEAM_SPREAD,GAME_TOTAL,IS_PLAYOFF
0,Stephen Curry,2021-05-16,GSW vs. MEM,46.0,-3.5,228.0,0
1,Jordan Nwora,2021-05-16,MIL @ CHI,34.0,-7.0,224.5,0
2,Moses Brown,2021-05-16,OKC vs. LAC,24.0,8.0,220.5,0
3,Domantas Sabonis,2021-05-16,IND @ TOR,25.0,-6.0,227.0,0
4,Russell Westbrook,2021-05-16,WAS vs. CHA,23.0,-6.5,230.5,0


In [76]:
if UPLOAD_TO_SUPABASE:
    upsert_silver(df, season_type=SEASON_TYPE)
else:
    print(f"Skipping upload — {len(df):,} rows in memory")

    … 10,000/23,054 rows
    … 20,000/23,054 rows
    … 23,054/23,054 rows
  ✓ silver.player_gamelogs — 23,054 rows upserted (postgres)


In [77]:
from src.utils.db import read_df

check = read_df(
    "player_gamelogs",
    schema="silver",
    where="season_year = %(s)s AND season_type = %(st)s",
    params={"s": SEASON, "st": SEASON_TYPE},
)
print(f"silver.player_gamelogs — {len(check):,} rows")
print(check["is_playoff"].value_counts())
check[["player_name", "game_date", "pts", "team_spread", "is_playoff"]].head()

silver.player_gamelogs — 23,054 rows
is_playoff
0    23054
Name: count, dtype: int64


,player_name,game_date,pts,team_spread,is_playoff
0,Torrey Craig,2021-05-15,11.0,-10.5,0
1,Nemanja Bjelica,2021-05-15,11.0,-3.0,0
2,Brook Lopez,2021-05-15,18.0,3.0,0
3,Duncan Robinson,2021-05-15,17.0,-3.0,0
4,Stephen Curry,2021-05-16,46.0,-3.5,0


In [71]:
from src.utils.gold import build_min_gold_from_silver
from src.utils.db import upsert_gold

SEASON_JOBS = [
    ("2024-25", "S25"),
    ("2023-24", "S24"),
    ("2022-23", "S23"),
    ("2021-22", "S22"),
    # add these only if you have silver + want them in the MIN model
    # ("2020-21", "S21"),
    # ("2019-20", "S20"),
]

for season_year, season_label in SEASON_JOBS:
    gold_df = build_min_gold_from_silver(
        season_year, "Regular Season", season_label=season_label
    )
    if gold_df.empty:
        print(f"skip {season_year} — no silver rows")
        continue
    print(f"{season_year} ({season_label}) — {len(gold_df):,} rows")
    upsert_gold(gold_df)

c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\nba_model\lib\site-packages\executing\executing.py:466: RuntimeWarning: coroutine 'run_scrape' was never awaited
  return compile(


KeyboardInterrupt: 

In [2]:
from src.utils.gold import build_ppm_gold_from_silver
from src.utils.db import upsert_ppm_gold, read_df

SEASON_TYPE = "Regular Season"

SEASON_JOBS = [
    ("2024-25", "S25"),
    ("2023-24", "S24"),
    ("2022-23", "S23"),
    ("2021-22", "S22"),
    ("2025-26", "S26"),
]

for season_year, season_label in SEASON_JOBS:
    gold_df = build_ppm_gold_from_silver(
        season_year, SEASON_TYPE, season_label=season_label
    )
    if gold_df.empty:
        print(f"skip {season_year} — no silver rows")
        continue
    print(f"{season_year} ({season_label}) — {len(gold_df):,} rows")
    upsert_ppm_gold(gold_df)

# verify
check = read_df("player_ppm_model", schema="gold", where="season = %(s)s", params={"s": "S25"})
print(f"gold.player_ppm_model (S25) — {len(check):,} rows")

2024-25 (S25) — 26,306 rows
  → dropping 1 column(s) not in gold.player_ppm_model: ['cfga_per_min_x_opp_fg_pct_allowed']
    … 10,000/26,306 rows
    … 20,000/26,306 rows
    … 26,306/26,306 rows
  ✓ gold.player_ppm_model — 26,306 rows upserted (postgres)
2023-24 (S24) — 26,401 rows
  → dropping 1 column(s) not in gold.player_ppm_model: ['cfga_per_min_x_opp_fg_pct_allowed']
    … 10,000/26,401 rows
    … 20,000/26,401 rows
    … 26,401/26,401 rows
  ✓ gold.player_ppm_model — 26,401 rows upserted (postgres)
2022-23 (S23) — 25,894 rows
  → dropping 1 column(s) not in gold.player_ppm_model: ['cfga_per_min_x_opp_fg_pct_allowed']
    … 10,000/25,894 rows
    … 20,000/25,894 rows
    … 25,894/25,894 rows
  ✓ gold.player_ppm_model — 25,894 rows upserted (postgres)
2021-22 (S22) — 26,039 rows
  → dropping 1 column(s) not in gold.player_ppm_model: ['cfga_per_min_x_opp_fg_pct_allowed']
    … 10,000/26,039 rows
    … 20,000/26,039 rows
    … 26,039/26,039 rows
  ✓ gold.player_ppm_model — 26,039 r